# Plot Correlation MC

Load the saved NetCDF correlation caches produced by the split build notebooks, then render the same maps without recomputing the DJF correlations.

# 1. Import Library

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

from scipy.ndimage import gaussian_filter
plt.rcParams.update({
    'font.family': 'Helvetica',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
})

CACHE_DIR = Path('../../data/intermediate/divided_correlation')
RESULTS_DIR = Path('../../results/analisis_korelasi_26-19')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def save_plot(fig, filename):
    fig.savefig(RESULTS_DIR / filename, dpi=300, bbox_inches='tight')

def smooth_for_plot(da, sigma=0.7):
    if not {'lat', 'lon'}.issubset(da.dims):
        return da
    def _smooth(arr):
        return gaussian_filter(arr, sigma=sigma)
    smoothed = xr.apply_ufunc(
        _smooth,
        da,
        input_core_dims=[['lat', 'lon']],
        output_core_dims=[['lat', 'lon']],
        vectorize=True,
        dask='allowed',
        output_dtypes=[da.dtype],
    )
    smoothed = smoothed.transpose(*da.dims)
    smoothed = smoothed.assign_attrs(da.attrs)
    if da.name is not None:
        smoothed = smoothed.rename(da.name)
    return smoothed

corr_levels = np.arange(-1, 1.01, 0.05)
corr_ticks = np.arange(-1, 1.01, 0.25)
wind_step = 8
wind_scale = 30
wind_diff_scale = 15

mc_extent = [80, 160, -20, 20]
mc_xticks = np.arange(90, 161, 20)
mc_yticks = np.arange(-20, 21, 10)


# 2. Load Cache

In [ ]:
rain_ds = xr.open_dataset(CACHE_DIR / 'rainfall_corr_cache_1981_2025.nc')
wind_ds = xr.open_dataset(CACHE_DIR / 'wind_corr_cache_1981_2025.nc')
mfc_ds = xr.open_dataset(CACHE_DIR / 'mfc_corr_cache_1981_2025.nc')
psi_ds = xr.open_dataset(CACHE_DIR / 'psi_corr_cache_1981_2025.nc')
chi_ds = xr.open_dataset(CACHE_DIR / 'chi_corr_cache_1981_2025.nc')
mslp_ds = xr.open_dataset(CACHE_DIR / 'mslp_corr_cache_1981_2025.nc')
gh850_ds = xr.open_dataset(CACHE_DIR / 'gh850_corr_cache_1981_2025.nc')

rain_corr_clim = rain_ds['rain_corr_clim']
rain_corr_past = rain_ds['rain_corr_past']
rain_corr_recent = rain_ds['rain_corr_recent']
rain_corr_recent_minus_past = rain_ds['rain_corr_recent_minus_past']
rain_corr_clim_sig = rain_ds['rain_corr_clim_sig']
rain_corr_past_sig = rain_ds['rain_corr_past_sig']
rain_corr_recent_sig = rain_ds['rain_corr_recent_sig']
rain_corr_recent_minus_past_sig = rain_ds['rain_corr_recent_minus_past_sig']

wind_u_corr_clim = wind_ds['wind_u_corr_clim']
wind_u_corr_past = wind_ds['wind_u_corr_past']
wind_u_corr_recent = wind_ds['wind_u_corr_recent']
wind_u_corr_recent_minus_past = wind_ds['wind_u_corr_recent_minus_past']
wind_u_corr_clim_sig = wind_ds['wind_u_corr_clim_sig']
wind_u_corr_past_sig = wind_ds['wind_u_corr_past_sig']
wind_u_corr_recent_sig = wind_ds['wind_u_corr_recent_sig']

wind_v_corr_clim = wind_ds['wind_v_corr_clim']
wind_v_corr_past = wind_ds['wind_v_corr_past']
wind_v_corr_recent = wind_ds['wind_v_corr_recent']
wind_v_corr_recent_minus_past = wind_ds['wind_v_corr_recent_minus_past']
wind_v_corr_clim_sig = wind_ds['wind_v_corr_clim_sig']
wind_v_corr_past_sig = wind_ds['wind_v_corr_past_sig']
wind_v_corr_recent_sig = wind_ds['wind_v_corr_recent_sig']
wind_vector_sig_clim = wind_ds['wind_vector_sig_clim']
wind_vector_sig_past = wind_ds['wind_vector_sig_past']
wind_vector_sig_recent = wind_ds['wind_vector_sig_recent']
wind_vector_sig_recent_minus_past = wind_ds['wind_vector_sig_recent_minus_past']
wind_u_corr_recent_minus_past_sig = wind_vector_sig_recent_minus_past

mfc_corr_clim = mfc_ds['mfc_corr_clim']
mfc_corr_past = mfc_ds['mfc_corr_past']
mfc_corr_recent = mfc_ds['mfc_corr_recent']
mfc_corr_recent_minus_past = mfc_ds['mfc_corr_recent_minus_past']
mfc_corr_clim_sig = mfc_ds['mfc_corr_clim_sig']
mfc_corr_past_sig = mfc_ds['mfc_corr_past_sig']
mfc_corr_recent_sig = mfc_ds['mfc_corr_recent_sig']
mfc_corr_recent_minus_past_sig = mfc_ds['mfc_corr_recent_minus_past_sig']
mfc_qx_corr_clim = mfc_ds['mfc_qx_corr_clim']
mfc_qx_corr_past = mfc_ds['mfc_qx_corr_past']
mfc_qx_corr_recent = mfc_ds['mfc_qx_corr_recent']
mfc_qx_corr_recent_minus_past = mfc_ds['mfc_qx_corr_recent_minus_past']
mfc_qx_corr_clim_sig = mfc_ds['mfc_qx_corr_clim_sig']
mfc_qx_corr_past_sig = mfc_ds['mfc_qx_corr_past_sig']
mfc_qx_corr_recent_sig = mfc_ds['mfc_qx_corr_recent_sig']
mfc_qy_corr_clim = mfc_ds['mfc_qy_corr_clim']
mfc_qy_corr_past = mfc_ds['mfc_qy_corr_past']
mfc_qy_corr_recent = mfc_ds['mfc_qy_corr_recent']
mfc_qy_corr_recent_minus_past = mfc_ds['mfc_qy_corr_recent_minus_past']
mfc_qy_corr_clim_sig = mfc_ds['mfc_qy_corr_clim_sig']
mfc_qy_corr_past_sig = mfc_ds['mfc_qy_corr_past_sig']
mfc_qy_corr_recent_sig = mfc_ds['mfc_qy_corr_recent_sig']
mfc_vector_sig_clim = mfc_ds['mfc_vector_sig_clim']
mfc_vector_sig_past = mfc_ds['mfc_vector_sig_past']
mfc_vector_sig_recent = mfc_ds['mfc_vector_sig_recent']
mfc_vector_sig_recent_minus_past = mfc_ds['mfc_vector_sig_recent_minus_past']

psi_corr_clim = psi_ds['psi_corr_clim']
psi_corr_past = psi_ds['psi_corr_past']
psi_corr_recent = psi_ds['psi_corr_recent']
psi_corr_recent_minus_past = psi_ds['psi_corr_recent_minus_past']
psi_corr_clim_sig = psi_ds['psi_corr_clim_sig']
psi_corr_past_sig = psi_ds['psi_corr_past_sig']
psi_corr_recent_sig = psi_ds['psi_corr_recent_sig']
psi_corr_recent_minus_past_sig = psi_ds['psi_corr_recent_minus_past_sig']
u_psi_corr_clim = psi_ds['u_psi_corr_clim']
u_psi_corr_past = psi_ds['u_psi_corr_past']
u_psi_corr_recent = psi_ds['u_psi_corr_recent']
u_psi_corr_recent_minus_past = psi_ds['u_psi_corr_recent_minus_past']
v_psi_corr_clim = psi_ds['v_psi_corr_clim']
v_psi_corr_past = psi_ds['v_psi_corr_past']
v_psi_corr_recent = psi_ds['v_psi_corr_recent']
v_psi_corr_recent_minus_past = psi_ds['v_psi_corr_recent_minus_past']
psi_vector_sig_clim = psi_ds['psi_vector_sig_clim']
psi_vector_sig_past = psi_ds['psi_vector_sig_past']
psi_vector_sig_recent = psi_ds['psi_vector_sig_recent']
psi_vector_sig_recent_minus_past = psi_ds['psi_vector_sig_recent_minus_past']

chi_corr_clim = chi_ds['chi_corr_clim']
chi_corr_past = chi_ds['chi_corr_past']
chi_corr_recent = chi_ds['chi_corr_recent']
chi_corr_recent_minus_past = chi_ds['chi_corr_recent_minus_past']
chi_corr_clim_sig = chi_ds['chi_corr_clim_sig']
chi_corr_past_sig = chi_ds['chi_corr_past_sig']
chi_corr_recent_sig = chi_ds['chi_corr_recent_sig']
chi_corr_recent_minus_past_sig = chi_ds['chi_corr_recent_minus_past_sig']
u_chi_corr_clim = chi_ds['u_chi_corr_clim']
u_chi_corr_past = chi_ds['u_chi_corr_past']
u_chi_corr_recent = chi_ds['u_chi_corr_recent']
u_chi_corr_recent_minus_past = chi_ds['u_chi_corr_recent_minus_past']
v_chi_corr_clim = chi_ds['v_chi_corr_clim']
v_chi_corr_past = chi_ds['v_chi_corr_past']
v_chi_corr_recent = chi_ds['v_chi_corr_recent']
v_chi_corr_recent_minus_past = chi_ds['v_chi_corr_recent_minus_past']
chi_vector_sig_clim = chi_ds['chi_vector_sig_clim']
chi_vector_sig_past = chi_ds['chi_vector_sig_past']
chi_vector_sig_recent = chi_ds['chi_vector_sig_recent']
chi_vector_sig_recent_minus_past = chi_ds['chi_vector_sig_recent_minus_past']

mslp_corr_clim = mslp_ds['mslp_corr_clim']
mslp_corr_past = mslp_ds['mslp_corr_past']
mslp_corr_recent = mslp_ds['mslp_corr_recent']
mslp_corr_recent_minus_past = mslp_ds['mslp_corr_recent_minus_past']
mslp_corr_clim_sig = mslp_ds['mslp_corr_clim_sig']
mslp_corr_past_sig = mslp_ds['mslp_corr_past_sig']
mslp_corr_recent_sig = mslp_ds['mslp_corr_recent_sig']
mslp_corr_recent_minus_past_sig = mslp_ds['mslp_corr_recent_minus_past_sig']

gh850_corr_clim = gh850_ds['gh850_corr_clim']
gh850_corr_past = gh850_ds['gh850_corr_past']
gh850_corr_recent = gh850_ds['gh850_corr_recent']
gh850_corr_recent_minus_past = gh850_ds['gh850_corr_recent_minus_past']
gh850_corr_clim_sig = gh850_ds['gh850_corr_clim_sig']
gh850_corr_past_sig = gh850_ds['gh850_corr_past_sig']
gh850_corr_recent_sig = gh850_ds['gh850_corr_recent_sig']
gh850_corr_recent_minus_past_sig = gh850_ds['gh850_corr_recent_minus_past_sig']


# 3. Rainfall Correlation

In [ ]:
plot_defs = [
    (rain_corr_clim, rain_corr_clim_sig, 'Korelasi Rainfall vs Nino34 DJF 1981-2025', 'rainfall_corr_1981-2025.png'),
    (rain_corr_past, rain_corr_past_sig, 'Korelasi Rainfall vs Nino34 DJF P1', 'rainfall_corr_p1.png'),
    (rain_corr_recent, rain_corr_recent_sig, 'Korelasi Rainfall vs Nino34 DJF P2', 'rainfall_corr_p2.png'),
    (rain_corr_recent_minus_past, rain_corr_recent_minus_past_sig, 'Selisih Korelasi Rainfall P2-P1', 'rainfall_corr_p2_minus_p1.png'),
]

for data, sig_mask, title, filename in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = smooth_for_plot(data).reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        add_labels=False,
        infer_intervals=True,
    )

    sig_plot = (sig_mask.fillna(False).astype(np.int8).coarsen(lat=8, lon=8, boundary='trim').max() > 0)
    yy, xx = np.where(sig_plot.values)
    if yy.size > 0:
        ax.scatter(
            sig_plot['lon'].values[xx],
            sig_plot['lat'].values[yy],
            s=15,
            c='k',
            marker='.',
            linewidths=0,
            alpha=0.7,
            transform=ccrs.PlateCarree(),
            zorder=4,
        )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi rainfall', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


# 4. Wind (vector) - U Shade


In [ ]:
plot_defs = [
    (wind_u_corr_clim, wind_u_corr_clim, wind_v_corr_clim, 'Korelasi UWND vs Nino34 DJF 1981-2025', 'wind_u_corr_1981-2025.png', wind_scale, 1.0, 'corr = 1.0'),
    (wind_u_corr_past, wind_u_corr_past, wind_v_corr_past, 'Korelasi UWND vs Nino34 DJF P1', 'wind_u_corr_p1.png', wind_scale, 1.0, 'corr = 1.0'),
    (wind_u_corr_recent, wind_u_corr_recent, wind_v_corr_recent, 'Korelasi UWND vs Nino34 DJF P2', 'wind_u_corr_p2.png', wind_scale, 1.0, 'corr = 1.0'),
    (wind_u_corr_recent_minus_past, wind_u_corr_recent_minus_past, wind_v_corr_recent_minus_past, 'Selisih Korelasi UWND P2-P1', 'wind_u_corr_p2_minus_p1.png', wind_scale, 1.0, 'corr = 1.0'),
]

for shade_data, u_data, v_data, title, filename, scale, key_u, key_label in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = smooth_for_plot(shade_data)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        add_labels=False,
        infer_intervals=True,
    )

    u_plot = u_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    v_plot = v_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=0.002,
        color='black',
        zorder=4,
    )
    ax.quiverkey(q, X=0.88, Y=1.06, U=key_u, label=key_label, labelpos='E', coordinates='axes', color='black')

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi u-wind', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


# 5. Wind (vector) - V Shade


In [ ]:
plot_defs = [
    (wind_v_corr_clim, wind_u_corr_clim, wind_v_corr_clim, 'Korelasi VWND vs Nino34 DJF 1981-2025', 'wind_v_corr_1981-2025.png', wind_scale, 1.0, 'corr = 1.0'),
    (wind_v_corr_past, wind_u_corr_past, wind_v_corr_past, 'Korelasi VWND vs Nino34 DJF P1', 'wind_v_corr_p1.png', wind_scale, 1.0, 'corr = 1.0'),
    (wind_v_corr_recent, wind_u_corr_recent, wind_v_corr_recent, 'Korelasi VWND vs Nino34 DJF P2', 'wind_v_corr_p2.png', wind_scale, 1.0, 'corr = 1.0'),
    (wind_v_corr_recent_minus_past, wind_u_corr_recent_minus_past, wind_v_corr_recent_minus_past, 'Selisih Korelasi VWND P2-P1', 'wind_v_corr_p2_minus_p1.png', wind_scale, 1.0, 'corr = 1.0'),
]

for shade_data, u_data, v_data, title, filename, scale, key_u, key_label in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = shade_data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        add_labels=False,
        infer_intervals=True,
    )

    u_plot = u_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    v_plot = v_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=0.002,
        color='black',
        zorder=4,
    )
    ax.quiverkey(q, X=0.88, Y=1.06, U=key_u, label=key_label, labelpos='E', coordinates='axes', color='black')

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi v-wind', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


# 6. MFC Correlation


In [ ]:
plot_defs = [
    (mfc_corr_clim, mfc_qx_corr_clim, mfc_qy_corr_clim, 'Korelasi MFC vs Nino34 DJF 1981-2025', 'mfc_corr_1981-2025.png', wind_scale, 1.0, 'corr = 1.0'),
    (mfc_corr_past, mfc_qx_corr_past, mfc_qy_corr_past, 'Korelasi MFC vs Nino34 DJF P1', 'mfc_corr_p1.png', wind_scale, 1.0, 'corr = 1.0'),
    (mfc_corr_recent, mfc_qx_corr_recent, mfc_qy_corr_recent, 'Korelasi MFC vs Nino34 DJF P2', 'mfc_corr_p2.png', wind_scale, 1.0, 'corr = 1.0'),
    (mfc_corr_recent_minus_past, mfc_qx_corr_recent_minus_past, mfc_qy_corr_recent_minus_past, 'Selisih Korelasi MFC P2-P1', 'mfc_corr_p2_minus_p1.png', wind_scale, 1.0, 'corr = 1.0'),
]

for shade_data, u_data, v_data, title, filename, scale, key_u, key_label in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = smooth_for_plot(shade_data)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        add_labels=False,
        infer_intervals=True,
    )

    u_plot = u_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    v_plot = v_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=0.002,
        color='black',
        zorder=4,
    )
    ax.quiverkey(q, X=0.88, Y=1.06, U=key_u, label=key_label, labelpos='E', coordinates='axes', color='black')

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi MFC', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


# 7. Streamfunction Correlation


In [ ]:
plot_defs = [
    (psi_corr_clim, u_psi_corr_clim, v_psi_corr_clim, 'Korelasi Streamfunction vs Nino34 DJF 1981-2025', 'streamfunction_corr_1981-2025.png', wind_scale, 1.0, 'corr = 1.0'),
    (psi_corr_past, u_psi_corr_past, v_psi_corr_past, 'Korelasi Streamfunction vs Nino34 DJF P1', 'streamfunction_corr_p1.png', wind_scale, 1.0, 'corr = 1.0'),
    (psi_corr_recent, u_psi_corr_recent, v_psi_corr_recent, 'Korelasi Streamfunction vs Nino34 DJF P2', 'streamfunction_corr_p2.png', wind_scale, 1.0, 'corr = 1.0'),
    (psi_corr_recent_minus_past, u_psi_corr_recent_minus_past, v_psi_corr_recent_minus_past, 'Selisih Korelasi Streamfunction P2-P1', 'streamfunction_corr_p2_minus_p1.png', wind_scale, 1.0, 'corr = 1.0'),
]

for shade_data, u_data, v_data, title, filename, scale, key_u, key_label in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = shade_data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    u_plot = u_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    v_plot = v_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=0.002,
        color='black',
        zorder=4,
    )
    ax.quiverkey(q, X=0.88, Y=1.06, U=key_u, label=key_label, labelpos='E', coordinates='axes', color='black')

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi streamfunction', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


# 8. Velocity Potential Correlation


In [ ]:
plot_defs = [
    (chi_corr_clim, u_chi_corr_clim, v_chi_corr_clim, 'Korelasi Velocity Potential vs Nino34 DJF 1981-2025', 'velocity_potential_corr_1981-2025.png', wind_scale, 1.0, 'corr = 1.0'),
    (chi_corr_past, u_chi_corr_past, v_chi_corr_past, 'Korelasi Velocity Potential vs Nino34 DJF P1', 'velocity_potential_corr_p1.png', wind_scale, 1.0, 'corr = 1.0'),
    (chi_corr_recent, u_chi_corr_recent, v_chi_corr_recent, 'Korelasi Velocity Potential vs Nino34 DJF P2', 'velocity_potential_corr_p2.png', wind_scale, 1.0, 'corr = 1.0'),
    (chi_corr_recent_minus_past, u_chi_corr_recent_minus_past, v_chi_corr_recent_minus_past, 'Selisih Korelasi Velocity Potential P2-P1', 'velocity_potential_corr_p2_minus_p1.png', wind_scale, 1.0, 'corr = 1.0'),
]

for shade_data, u_data, v_data, title, filename, scale, key_u, key_label in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = shade_data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    u_plot = u_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    v_plot = v_data.isel(lat=slice(None, None, wind_step), lon=slice(None, None, wind_step))
    q = ax.quiver(
        u_plot['lon'].values,
        u_plot['lat'].values,
        u_plot.values,
        v_plot.values,
        transform=ccrs.PlateCarree(),
        scale=scale,
        width=0.002,
        color='black',
        zorder=4,
    )
    ax.quiverkey(q, X=0.88, Y=1.06, U=key_u, label=key_label, labelpos='E', coordinates='axes', color='black')

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi velocity potential', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


# 9. MSLP Correlation

In [ ]:
plot_defs = [
    (mslp_corr_clim, mslp_corr_clim_sig, 'Korelasi MSLP vs Nino34 DJF 1981-2025', 'mslp_corr_1981-2025.png'),
    (mslp_corr_past, mslp_corr_past_sig, 'Korelasi MSLP vs Nino34 DJF P1', 'mslp_corr_p1.png'),
    (mslp_corr_recent, mslp_corr_recent_sig, 'Korelasi MSLP vs Nino34 DJF P2', 'mslp_corr_p2.png'),
    (mslp_corr_recent_minus_past, mslp_corr_recent_minus_past_sig, 'Selisih Korelasi MSLP P2-P1', 'mslp_corr_p2_minus_p1.png'),
]

for data, sig_mask, title, filename in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    sig_plot = (sig_mask.fillna(False).astype(np.int8).coarsen(lat=8, lon=8, boundary='trim').max() > 0)
    yy, xx = np.where(sig_plot.values)
    if yy.size > 0:
        ax.scatter(
            sig_plot['lon'].values[xx],
            sig_plot['lat'].values[yy],
            s=15,
            c='k',
            marker='.',
            linewidths=0,
            alpha=0.7,
            transform=ccrs.PlateCarree(),
            zorder=4,
        )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi mslp', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)


# 10. Geopotential Height 850 hPa Correlation

In [ ]:
plot_defs = [
    (gh850_corr_clim, gh850_corr_clim_sig, 'Korelasi Geopotential Height 850 hPa vs Nino34 DJF 1981-2025', 'geopotential_height_850_corr_1981-2025.png'),
    (gh850_corr_past, gh850_corr_past_sig, 'Korelasi Geopotential Height 850 hPa vs Nino34 DJF P1', 'geopotential_height_850_corr_p1.png'),
    (gh850_corr_recent, gh850_corr_recent_sig, 'Korelasi Geopotential Height 850 hPa vs Nino34 DJF P2', 'geopotential_height_850_corr_p2.png'),
    (gh850_corr_recent_minus_past, gh850_corr_recent_minus_past_sig, 'Selisih Korelasi Geopotential Height 850 hPa P2-P1', 'geopotential_height_850_corr_p2_minus_p1.png'),
]

for data, sig_mask, title, filename in plot_defs:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree(central_longitude=180))

    plot_data = data.reset_coords(drop=True)
    img = plot_data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='bwr',
        levels=corr_levels,
        extend='both',
        add_colorbar=False,
        infer_intervals=True,
    )

    sig_plot = (sig_mask.fillna(False).astype(np.int8).coarsen(lat=8, lon=8, boundary='trim').max() > 0)
    yy, xx = np.where(sig_plot.values)
    if yy.size > 0:
        ax.scatter(
            sig_plot['lon'].values[xx],
            sig_plot['lat'].values[yy],
            s=15,
            c='k',
            marker='.',
            linewidths=0,
            alpha=0.7,
            transform=ccrs.PlateCarree(),
            zorder=4,
        )

    ax.coastlines(resolution='10m', linewidth=0.6, color='black', zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.4, color='black', zorder=3)
    ax.set_extent(mc_extent, crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
    ax.set_xticks(mc_xticks, crs=ccrs.PlateCarree())
    ax.set_yticks(mc_yticks, crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(direction='out', top=True, right=True, labelsize=14)
    ax.set_title(title, fontsize=14, fontweight='bold', loc='left', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('')

    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.15, ax.get_position().width, 0.04])
    cbar = fig.colorbar(img, cax=cax, orientation='horizontal', ticks=corr_ticks, spacing='proportional', extend='both')
    cbar.set_label('Koefisien korelasi geopotential height 850 hPa', fontsize=14, labelpad=10)
    cbar.ax.tick_params(labelsize=14)

    save_plot(fig, filename)
    plt.show()
    plt.close(fig)
